# Phase 6 (extension) - Algorithmic fog vs GAN-generated fog

Our main pipeline used Pix2Pix trained on Foggy Cityscapes (street-level) to
generate foggy versions of VDD (aerial drone images). This introduces a
**domain gap**: the GAN learned fog patterns on a different visual domain.

This extension answers: **what if we use a simple algorithmic fog (Albumentations)
directly on VDD, without any GAN?**

## Methods compared

| Method     | How VDD_foggy is generated                              |
|------------|---------------------------------------------------------|
| **gan**    | Pix2Pix trained on Foggy Cityscapes, applied to VDD     |
| **alg**    | Albumentations.RandomFog applied directly to VDD        |

## Pipeline of this notebook

1. Verify GPU
2. Clone code from GitHub
3. Mount Google Drive
4. Restore VDD + the U-Net v2/v3 checkpoints + foggy_gan datasets
5. Install dependencies
6. Generate VDD_foggy_alg_medium and VDD_foggy_alg_dense (fast, CPU)
7. Visual comparison: clean / GAN-medium / alg-medium / GAN-dense / alg-dense
8. Evaluate U-Net v2 on the 2 algorithmic foggy variants
9. Re-train U-Net v3_alg on mixed (clean + alg_medium + alg_dense)
10. Evaluate v3_alg on all test sets (clean, gan_medium, gan_dense, alg_medium, alg_dense)
11. Final comparison table (v2 vs v3_gan vs v3_alg)
12. Backup results to Drive

**Time**: ~60 minutes total on T4 (most of it is the v3_alg retraining).

**Before running**: Runtime -> Change runtime type -> T4 GPU.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone project + mount Drive

In [ ]:
import os

if not os.path.isdir('/content/fog-uav-robustness'):
    !git clone https://github.com/FabrizioCozzolino/fog-uav-robustness.git /content/fog-uav-robustness
%cd /content/fog-uav-robustness
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/fog-uav-robustness'

## 3. Restore everything we need from Drive

Required:
  - VDD (the clean source)
  - U-Net v2 checkpoint (the baseline to evaluate on alg foggy)
  - VDD_foggy_medium_768 + VDD_foggy_dense_768 (the GAN-generated foggy, for v3_gan comparison)
  - U-Net v3 (the GAN-trained mixed model, already obtained in Phase 5)

If your previous Colab session is still alive, this cell skips everything.

In [ ]:
import os, time, shutil

# === Datasets ===
DATASETS_NEEDED = ['VDD', 'VDD_foggy_medium_768', 'VDD_foggy_dense_768']
for ds in DATASETS_NEEDED:
    LOCAL = f'/content/data/{ds}'
    train_src = os.path.join(LOCAL, 'train', 'src')
    if os.path.isdir(train_src) and len(os.listdir(train_src)) >= 280:
        print(f'[ok] {ds} already at {LOCAL}')
        continue
    DRIVE_DS = f'{DRIVE}/data/{ds}'
    if ds == 'VDD' and os.path.isdir(os.path.join(DRIVE_DS, 'VDD', 'train')):
        SRC = os.path.join(DRIVE_DS, 'VDD')
    elif os.path.isdir(os.path.join(DRIVE_DS, 'train')):
        SRC = DRIVE_DS
    else:
        raise RuntimeError(f'Cannot find train/ inside {DRIVE_DS}')
    if os.path.isdir(LOCAL):
        shutil.rmtree(LOCAL)
    os.makedirs('/content/data', exist_ok=True)
    print(f'Copying {SRC} -> {LOCAL}')
    t0 = time.time()
    !cp -r "{SRC}" "{LOCAL}"
    print(f'  done in {time.time() - t0:.0f} s')

# === U-Net checkpoints ===
for run_name in ['unet_resnet34_clean_v2_weighted', 'unet_resnet34_mixed_v3']:
    SRC = f'{DRIVE}/outputs/{run_name}'
    DST = f'/content/fog-uav-robustness/outputs/runs/{run_name}'
    if os.path.isfile(f'{DST}/best.pth'):
        print(f'[ok] {run_name} already restored')
        continue
    if not os.path.isdir(SRC):
        raise RuntimeError(f'{run_name} not on Drive!')
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copytree(SRC, DST)
    print(f'[ok] restored {run_name}')

print('\n=== Summary ===')
for ds in DATASETS_NEEDED:
    LOCAL = f'/content/data/{ds}'
    counts = {split: len(os.listdir(f'{LOCAL}/{split}/src')) for split in ['train','val','test']}
    print(f'  {ds}: train={counts["train"]}, val={counts["val"]}, test={counts["test"]}')

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## 5. Generate algorithmic foggy datasets (medium + dense)

Fast (~2 minutes total): the Albumentations transform is CPU-bound but
lightweight. Output: 400 images each.

In [ ]:
!python src/inference/generate_foggy_vdd_alg.py \
    --vdd-root /content/data/VDD \
    --output-root /content/data/VDD_foggy_alg_medium_768 \
    --intensity medium \
    --save-size 768 \
    --seed 42

In [ ]:
!python src/inference/generate_foggy_vdd_alg.py \
    --vdd-root /content/data/VDD \
    --output-root /content/data/VDD_foggy_alg_dense_768 \
    --intensity dense \
    --save-size 768 \
    --seed 42

## 6. Visual comparison: clean / GAN / Albumentations

Show side-by-side on a few test images. This is the **most important sanity
check**: do the two fog sources LOOK similar? Or are they very different?

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

TEST_NAMES = ['DJI_0009', 'DJI_0357', 'DJI_0469', 'DJI_0917']
VARIANTS = [
    ('clean',         '/content/data/VDD/test/src',                       '.JPG'),
    ('GAN medium',    '/content/data/VDD_foggy_medium_768/test/src',      '.JPG'),
    ('alg medium',    '/content/data/VDD_foggy_alg_medium_768/test/src',  '.JPG'),
    ('GAN dense',     '/content/data/VDD_foggy_dense_768/test/src',       '.JPG'),
    ('alg dense',     '/content/data/VDD_foggy_alg_dense_768/test/src',   '.JPG'),
]

fig, axes = plt.subplots(len(TEST_NAMES), len(VARIANTS), figsize=(4 * len(VARIANTS), 3 * len(TEST_NAMES)))
for i, name in enumerate(TEST_NAMES):
    for j, (title, folder, ext) in enumerate(VARIANTS):
        path = os.path.join(folder, name + ext)
        if not os.path.isfile(path):
            axes[i, j].set_title(f'{title}\n[missing]', fontsize=9)
            axes[i, j].axis('off')
            continue
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(img)
        if i == 0:
            axes[i, j].set_title(title, fontsize=10)
        if j == 0:
            axes[i, j].set_ylabel(name, fontsize=9)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

os.makedirs('outputs/figures', exist_ok=True)
out_path = 'outputs/figures/phase6_gan_vs_alg_comparison.png'
plt.tight_layout()
plt.savefig(out_path, dpi=80, bbox_inches='tight')
plt.show()
print(f'\nSaved: {out_path}')

## 7. Evaluate U-Net v2 (clean-trained) on algorithmic foggy

Quick: ~30 sec each.

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD_foggy_alg_medium_768 \
    --split test --image-size 768 --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_foggy_alg_medium_768.json \
    --no-tb

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD_foggy_alg_dense_768 \
    --split test --image-size 768 --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_foggy_alg_dense_768.json \
    --no-tb

## 8. Re-train U-Net v3_alg on mixed clean + alg_medium + alg_dense

Same hyperparameters as v3 (Phase 5), only the foggy data sources change.
Mixed training set: clean (280) + alg_medium (280) + alg_dense (280) = 840 images.

Estimated time: ~30-60 min on T4 (depends on disk I/O).

In [ ]:
!python src/training/train_unet.py \
    --data-root /content/data/VDD \
    --data-roots /content/data/VDD \
                 /content/data/VDD_foggy_alg_medium_768 \
                 /content/data/VDD_foggy_alg_dense_768 \
    --output-dir outputs/runs \
    --run-name unet_resnet34_mixed_v3_alg \
    --encoder resnet34 \
    --image-size 768 \
    --epochs 30 \
    --batch-size 4 \
    --val-batch-size 4 \
    --num-workers 2 \
    --lr 1e-4 \
    --weight-decay 1e-4 \
    --grad-clip 1.0 \
    --class-weights inverse_sqrt \
    --seed 42

## 9. Evaluate v3_alg on all 5 test sets

We test v3_alg on: clean / GAN-medium / GAN-dense / alg-medium / alg-dense.
Goal: see how well v3_alg generalizes ACROSS fog generation methods, not just
to the kind of fog it was trained on.

In [ ]:
test_configs = [
    ('clean',         '/content/data/VDD'),
    ('foggy_medium_768',     '/content/data/VDD_foggy_medium_768'),
    ('foggy_dense_768',      '/content/data/VDD_foggy_dense_768'),
    ('foggy_alg_medium_768', '/content/data/VDD_foggy_alg_medium_768'),
    ('foggy_alg_dense_768',  '/content/data/VDD_foggy_alg_dense_768'),
]
for label, root in test_configs:
    print(f'\n=== Eval v3_alg on {label} ===')
    !python src/evaluation/evaluate.py \
        --checkpoint outputs/runs/unet_resnet34_mixed_v3_alg/best.pth \
        --data-root "{root}" \
        --split test --image-size 768 --batch-size 4 \
        --output "outputs/runs/unet_resnet34_mixed_v3_alg/test_results_{label}.json" \
        --no-tb

## 10. Comprehensive comparison table

Build the final results table comparing all three U-Net configurations
(v2, v3_gan, v3_alg) on all five test sets.

In [ ]:
import json, os

RUN_V2  = '/content/fog-uav-robustness/outputs/runs/unet_resnet34_clean_v2_weighted'
RUN_V3  = '/content/fog-uav-robustness/outputs/runs/unet_resnet34_mixed_v3'
RUN_V3A = '/content/fog-uav-robustness/outputs/runs/unet_resnet34_mixed_v3_alg'

test_sets = [
    ('clean',              'test_results_clean.json'),
    ('GAN medium @768',    'test_results_foggy_medium_768.json'),
    ('GAN dense @768',     'test_results_foggy_dense_768.json'),
    ('alg medium @768',    'test_results_foggy_alg_medium_768.json'),
    ('alg dense @768',     'test_results_foggy_alg_dense_768.json'),
]

def load_or_none(path):
    return json.load(open(path)) if os.path.isfile(path) else None

def get_results(run_dir):
    return {label: load_or_none(f'{run_dir}/{fname}') for label, fname in test_sets}

r_v2  = get_results(RUN_V2)
r_v3  = get_results(RUN_V3)
r_v3a = get_results(RUN_V3A)

print('=' * 96)
print('PHASE 6 (extension): Algorithmic fog vs GAN fog')
print('=' * 96)
print(f'{"test set":22s}  {"v2 mIoU":>10s}  {"v3_gan mIoU":>12s}  {"v3_alg mIoU":>12s}  {"best":>10s}')
print('-' * 96)
for label, _ in test_sets:
    v2 = r_v2.get(label); v3 = r_v3.get(label); v3a = r_v3a.get(label)
    if v2 is None or v3 is None or v3a is None:
        print(f'{label:22s}  [missing data]')
        continue
    values = {'v2': v2['mIoU'], 'v3_gan': v3['mIoU'], 'v3_alg': v3a['mIoU']}
    best = max(values, key=values.get)
    print(f'{label:22s}  {v2["mIoU"]:>10.4f}  {v3["mIoU"]:>12.4f}  {v3a["mIoU"]:>12.4f}  {best:>10s}')

print()
print('=== Per-class IoU on GAN dense @768 (the hardest test) ===')
if r_v2['GAN dense @768'] and r_v3['GAN dense @768'] and r_v3a['GAN dense @768']:
    classes = list(r_v2['GAN dense @768']['per_class_iou'].keys())
    print(f'{"class":14s}  {"v2":>8s}  {"v3_gan":>8s}  {"v3_alg":>8s}')
    for cls in classes:
        v2v  = r_v2['GAN dense @768']['per_class_iou'][cls]
        v3v  = r_v3['GAN dense @768']['per_class_iou'][cls]
        v3av = r_v3a['GAN dense @768']['per_class_iou'][cls]
        print(f'{cls:14s}  {v2v:>8.4f}  {v3v:>8.4f}  {v3av:>8.4f}')

In [ ]:
# Final visualization: 3-way bar chart
import matplotlib.pyplot as plt
import numpy as np

if all(r is not None for r in [r_v2.get(l, None) for l, _ in test_sets] +
                              [r_v3.get(l, None) for l, _ in test_sets] +
                              [r_v3a.get(l, None) for l, _ in test_sets]):
    labels_short = ['clean', 'GAN med', 'GAN dns', 'alg med', 'alg dns']
    v2_miou  = [r_v2[l]['mIoU']  for l, _ in test_sets]
    v3_miou  = [r_v3[l]['mIoU']  for l, _ in test_sets]
    v3a_miou = [r_v3a[l]['mIoU'] for l, _ in test_sets]

    x = np.arange(len(labels_short))
    width = 0.27

    fig, ax = plt.subplots(figsize=(11, 5))
    b1 = ax.bar(x - width, v2_miou, width, label='U-Net v2 (clean train)', color='#7F8C8D')
    b2 = ax.bar(x,         v3_miou, width, label='U-Net v3 (GAN-augmented)', color='#27AE60')
    b3 = ax.bar(x + width, v3a_miou, width, label='U-Net v3_alg (Albumentations-augmented)', color='#E67E22')

    ax.set_ylabel('test mIoU')
    ax.set_title('Robustness comparison: clean vs GAN-augmented vs Albumentations-augmented')
    ax.set_xticks(x); ax.set_xticklabels(labels_short)
    ax.legend(loc='lower left', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1)

    for bars in [b1, b2, b3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    out = '/content/fog-uav-robustness/outputs/figures/phase6_three_way_comparison.png'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    plt.savefig(out, dpi=80, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {out}')

## 11. Backup everything to Drive

In [ ]:
import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/fog-uav-robustness/data'

# Backup the 2 new foggy datasets
for ds in ['VDD_foggy_alg_medium_768', 'VDD_foggy_alg_dense_768']:
    SRC = f'/content/data/{ds}'
    DST = f'{DRIVE_DATA}/{ds}'
    if os.path.isdir(SRC):
        print(f'Backing up {ds} ...')
        if os.path.isdir(DST):
            shutil.rmtree(DST)
        shutil.copytree(SRC, DST)
        print(f'  [ok]')

# Backup the v3_alg run
RUN_V3A = 'unet_resnet34_mixed_v3_alg'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN_V3A}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN_V3A}'
if os.path.isdir(SRC):
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'[ok] {RUN_V3A} backed up')

# Also backup the new v2 JSONs (test on alg foggy)
RUN_V2 = 'unet_resnet34_clean_v2_weighted'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN_V2}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN_V2}'
for fname in os.listdir(SRC):
    if fname.startswith('test_results_foggy_alg') and fname.endswith('.json'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(DST, fname))
        print(f'[ok] {fname}')

# Backup figures
for fig_name in ['phase6_gan_vs_alg_comparison.png', 'phase6_three_way_comparison.png']:
    SRC = f'/content/fog-uav-robustness/outputs/figures/{fig_name}'
    DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/figures/{fig_name}'
    if os.path.isfile(SRC):
        os.makedirs(os.path.dirname(DST), exist_ok=True)
        shutil.copy(SRC, DST)
        print(f'[ok] {fig_name}')

print('\nAll done!')

## What we learn from this extension

Now we can answer several scientific questions:

1. **How does the U-Net v2 (clean-trained) react to algorithmic fog vs GAN fog?**
   Look at the v2 column: are mIoU drops similar between `GAN dense @768` and
   `alg dense @768`? If yes, the two fog sources cause similar perceptual damage.
   If no, one source is more aggressive.

2. **Does training on Albumentations fog (v3_alg) recover robustness as well as
   training on GAN fog (v3_gan)?**
   Compare on the same test set. If `v3_alg` matches or beats `v3_gan` on the
   GAN test sets, then we don't need the GAN at all -- a simple augmentation is
   enough. If `v3_alg` is much worse, then the GAN's learned fog distribution
   adds real value.

3. **Cross-method generalization**: does a model trained on one fog type test
   well on the other? Compare v3_gan on alg test sets, and v3_alg on GAN test
   sets. If both generalize, then the fog representations are 'aligned'. If they
   don't, each method has its own bias.

This is solid material for the discussion section of the report: it shows
scientific rigor (testing alternative methods) and grounds the GAN approach
in a real comparison.